# FEVER Dataset Audit

## Objective

This notebook performs a systematic, read-only audit of the raw FEVER development dataset and its associated Wikipedia evidence files.

The audit examines file integrity, schema, labels, duplicates, evidence annotations, Wikipedia resolution, content quality and experiment eligibility.

The raw dataset will not be modified.

## 0. Audit Setup

This section locates the raw FEVER claims file and Wikipedia evidence files and confirms that the required data is available.

In [1]:
from pathlib import Path

start = Path.cwd()
project_root = next(
    (path for path in (start, *start.parents) if (path / "pyproject.toml").exists()),
    None,
)

assert project_root is not None, "Could not locate the project root"

fever_dev = project_root / "data/fever/raw/shared_task_dev.jsonl"
wiki_dir = project_root / "data/fever/raw/wiki-pages/wiki-pages"
wiki_files = sorted(wiki_dir.glob("wiki-*.jsonl"))

print("Project root:", project_root)
print("FEVER dev exists:", fever_dev.exists())
print("FEVER dev size (MB):", round(fever_dev.stat().st_size / 1_000_000, 2))
print("Wikipedia directory exists:", wiki_dir.exists())
print("Wikipedia files:", len(wiki_files))
print("First Wikipedia file:", wiki_files[0].name if wiki_files else "None")
print("Last Wikipedia file:", wiki_files[-1].name if wiki_files else "None")

Project root: /Users/shanusharma/Desktop/UTS/NLP/Project/llm-fact-checker
FEVER dev exists: True
FEVER dev size (MB): 4.35
Wikipedia directory exists: True
Wikipedia files: 109
First Wikipedia file: wiki-001.jsonl
Last Wikipedia file: wiki-109.jsonl


## 1. File Integrity

### 1.1 FEVER Development File

This check verifies that the development file is valid UTF-8, calculates its SHA-256 hash and checks whether every non-empty line contains valid JSON.

In [2]:
import hashlib
import json

raw_bytes = fever_dev.read_bytes()
dataset_hash = hashlib.sha256(raw_bytes).hexdigest()
lines = raw_bytes.decode("utf-8").splitlines()

records = []
blank_lines = []
invalid_json_lines = []

for line_number, line in enumerate(lines, start=1):
    if not line.strip():
        blank_lines.append(line_number)
        continue

    try:
        records.append(json.loads(line))
    except json.JSONDecodeError as error:
        invalid_json_lines.append(
            {
                "line_number": line_number,
                "error": str(error),
            }
        )

print("SHA-256:", dataset_hash)
print("Total physical lines:", len(lines))
print("Valid JSON records:", len(records))
print("Blank lines:", len(blank_lines))
print("Invalid JSON lines:", len(invalid_json_lines))

if invalid_json_lines:
    print("First invalid line:", invalid_json_lines[0])

SHA-256: e89865bfe1b4dd054e03dd57d7241a6fde24862905f31117cf0cd719f7c78df7
Total physical lines: 19998
Valid JSON records: 19998
Blank lines: 0
Invalid JSON lines: 0


### 1.2 Wikipedia File Inventory

This check verifies that the expected 109 Wikipedia files are present, correctly numbered and non-empty.

Full JSON parsing will be combined with the later evidence-resolution audit to avoid scanning the large Wikipedia collection twice.

In [3]:
expected_names = {f"wiki-{number:03d}.jsonl" for number in range(1, 110)}
actual_names = {path.name for path in wiki_files}

missing_files = sorted(expected_names - actual_names)
unexpected_files = sorted(actual_names - expected_names)
empty_files = [path.name for path in wiki_files if path.stat().st_size == 0]
total_size_gb = sum(path.stat().st_size for path in wiki_files) / 1_000_000_000

print("Expected files:", len(expected_names))
print("Files found:", len(actual_names))
print("Missing files:", missing_files)
print("Unexpected files:", unexpected_files)
print("Empty files:", empty_files)
print("Total Wikipedia size (GB):", round(total_size_gb, 2))

Expected files: 109
Files found: 109
Missing files: []
Unexpected files: []
Empty files: []
Total Wikipedia size (GB): 7.62


## 2. Schema Audit

### 2.1 Fields, Types and Missing Values

This check identifies the fields present in FEVER records, their data types, missing values, null values, empty values and schema variations.

In [4]:
from collections import Counter

field_names = sorted({field for record in records for field in record})
schema_patterns = Counter(tuple(sorted(record)) for record in records)

print("Fields found:", field_names)
print("Number of schema patterns:", len(schema_patterns))
print()

for field in field_names:
    values = [record[field] for record in records if field in record]

    missing_count = sum(field not in record for record in records)
    null_count = sum(value is None for value in values)
    empty_count = sum(
        isinstance(value, (str, list, dict)) and len(value) == 0
        for value in values
    )
    types = sorted({type(value).__name__ for value in values})

    print(
        f"{field:12} "
        f"present={len(values):5} "
        f"missing={missing_count:5} "
        f"null={null_count:5} "
        f"empty={empty_count:5} "
        f"types={types}"
    )

print("\nSchema patterns:")
for fields, count in schema_patterns.items():
    print(count, fields)

Fields found: ['claim', 'evidence', 'id', 'label', 'verifiable']
Number of schema patterns: 1

claim        present=19998 missing=    0 null=    0 empty=    0 types=['str']
evidence     present=19998 missing=    0 null=    0 empty=    0 types=['list']
id           present=19998 missing=    0 null=    0 empty=    0 types=['int']
label        present=19998 missing=    0 null=    0 empty=    0 types=['str']
verifiable   present=19998 missing=    0 null=    0 empty=    0 types=['str']

Schema patterns:
19998 ('claim', 'evidence', 'id', 'label', 'verifiable')


#### Result

**PASS:** All 19,998 FEVER records follow one consistent schema containing `id`, `claim`, `label`, `verifiable` and `evidence`.

No missing, null or empty top-level values were found, and all fields use consistent data types.

## 3. Label Audit

### 3.1 Label and Verifiability Distribution

This check counts each label and examines whether labels are consistently paired with their verifiability status.

In [5]:
label_counts = Counter(record["label"] for record in records)
verifiable_counts = Counter(record["verifiable"] for record in records)
label_pairs = Counter(
    (record["label"], record["verifiable"])
    for record in records
)

print("Label counts:")
for label, count in sorted(label_counts.items()):
    print(f"{label}: {count} ({count / len(records):.2%})")

print("\nVerifiability counts:")
for status, count in sorted(verifiable_counts.items()):
    print(f"{status}: {count} ({count / len(records):.2%})")

print("\nLabel and verifiability combinations:")
for pair, count in sorted(label_pairs.items()):
    print(pair, count)

Label counts:
NOT ENOUGH INFO: 6666 (33.33%)
REFUTES: 6666 (33.33%)
SUPPORTS: 6666 (33.33%)

Verifiability counts:
NOT VERIFIABLE: 6666 (33.33%)
VERIFIABLE: 13332 (66.67%)

Label and verifiability combinations:
('NOT ENOUGH INFO', 'NOT VERIFIABLE') 6666
('REFUTES', 'VERIFIABLE') 6666
('SUPPORTS', 'VERIFIABLE') 6666


A claim is **verifiable** when FEVER provides enough Wikipedia evidence to classify it as either `SUPPORTS` or `REFUTES`.

A claim is **not verifiable** when there is insufficient evidence, so it receives the `NOT ENOUGH INFO` label.

- **13,332 claims (66.67%)** are verifiable.
- **6,666 claims (33.33%)** are not verifiable.

Being verifiable does not mean that a claim is true. It only means there is enough evidence to judge it.

#### Result

**PASS:** The dataset is perfectly balanced across the three FEVER labels, with 6,666 records (33.33%) in each class.

All label and verifiability combinations are internally consistent:

- `SUPPORTS` → `VERIFIABLE`
- `REFUTES` → `VERIFIABLE`
- `NOT ENOUGH INFO` → `NOT VERIFIABLE`

This confirms structural consistency but does not independently verify the factual correctness of each label.

## 4. Duplicate Audit

### 4.1 Duplicate IDs and Claims

This check identifies repeated record IDs, exact duplicate claim texts and duplicate claims assigned conflicting labels.

In [6]:
from collections import defaultdict

id_counts = Counter(record["id"] for record in records)

claims = defaultdict(list)
for record in records:
    claims[record["claim"]].append(record)

duplicate_ids = {
    record_id: count
    for record_id, count in id_counts.items()
    if count > 1
}

duplicate_claims = {
    claim: group
    for claim, group in claims.items()
    if len(group) > 1
}

conflicting_claims = {
    claim: group
    for claim, group in duplicate_claims.items()
    if len({record["label"] for record in group}) > 1
}

print("Duplicate ID values:", len(duplicate_ids))
print("Exact duplicate claim texts:", len(duplicate_claims))
print(
    "Records involved in duplicate claims:",
    sum(len(group) for group in duplicate_claims.values())
)
print("Duplicate claims with conflicting labels:", len(conflicting_claims))

Duplicate ID values: 0
Exact duplicate claim texts: 335
Records involved in duplicate claims: 736
Duplicate claims with conflicting labels: 54


#### Result

**WARNING:** No duplicate record IDs were found. However, 335 exact claim texts appear more than once, involving 736 records.

Among these, 54 duplicated claim groups contain conflicting FEVER labels. These cases require further inspection because identical claim text would normally be expected to have the same label.

### Official Dataset Documentation

According to the official FEVER documentation, claims were generated from a June 2017 Wikipedia dump and independently labelled as `SUPPORTS`, `REFUTES` or `NOT ENOUGH INFO`.

The shared-task development file contains 19,998 claims and combines the development and test datasets originally used in the FEVER paper.

The FEVER paper reports that the dataset is not completely error-free. Its validation identified occasional incorrect labels, incomplete evidence, ambiguous entities and conflicting information between Wikipedia pages.

The official documentation does not state that duplicate claims with conflicting labels are intentional. Therefore, the conflicting duplicate groups will be investigated rather than automatically removed or treated as errors.

### 4.2 Conflicting Duplicate Claims

This check examines the label combinations and record details for duplicated claims that have been assigned different labels.

In [7]:
conflict_patterns = Counter(
    tuple(sorted({record["label"] for record in group}))
    for group in conflicting_claims.values()
)

print("Conflicting label patterns:")
for pattern, count in sorted(conflict_patterns.items()):
    print(pattern, count)

print("\nFirst 10 conflicting claims:")

for number, (claim, group) in enumerate(
    list(conflicting_claims.items())[:10],
    start=1,
):
    print(f"\n{number}. Claim: {claim}")

    for record in group:
        print(
            "   ID:", record["id"],
            "| Label:", record["label"],
            "| Verifiable:", record["verifiable"],
            "| Evidence sets:", len(record["evidence"]),
        )

Conflicting label patterns:
('NOT ENOUGH INFO', 'REFUTES') 37
('NOT ENOUGH INFO', 'REFUTES', 'SUPPORTS') 1
('NOT ENOUGH INFO', 'SUPPORTS') 14
('REFUTES', 'SUPPORTS') 2

First 10 conflicting claims:

1. Claim: Despicable Me 2 was directed by a lake.
   ID: 202783 | Label: REFUTES | Verifiable: VERIFIABLE | Evidence sets: 2
   ID: 202757 | Label: NOT ENOUGH INFO | Verifiable: NOT VERIFIABLE | Evidence sets: 1

2. Claim: Microbiologist research promotes information found in other fields.
   ID: 102451 | Label: NOT ENOUGH INFO | Verifiable: NOT VERIFIABLE | Evidence sets: 1
   ID: 4240 | Label: SUPPORTS | Verifiable: VERIFIABLE | Evidence sets: 1

3. Claim: Kenneth Lonergan is a songwriter.
   ID: 184080 | Label: NOT ENOUGH INFO | Verifiable: NOT VERIFIABLE | Evidence sets: 1
   ID: 184065 | Label: NOT ENOUGH INFO | Verifiable: NOT VERIFIABLE | Evidence sets: 1
   ID: 184049 | Label: REFUTES | Verifiable: VERIFIABLE | Evidence sets: 1
   ID: 184047 | Label: NOT ENOUGH INFO | Verifiable: NO

#### Conflict Summary

**WARNING:** All 54 duplicate claim groups contain inconsistent labels.

Of these, 52 groups include `NOT ENOUGH INFO` conflicting with at least one verifiable label. This suggests possible differences in evidence discovery or annotation decisions.

Three groups contain a direct disagreement between `SUPPORTS` and `REFUTES`, including one group containing all three labels. These are the highest-priority cases for evidence inspection.

No records will be removed or corrected until their referenced Wikipedia evidence has been validated.

In [8]:
direct_conflicts = {
    claim: group
    for claim, group in conflicting_claims.items()
    if {"SUPPORTS", "REFUTES"}.issubset(
        {record["label"] for record in group}
    )
}

print("Groups containing both SUPPORTS and REFUTES:", len(direct_conflicts))

for number, (claim, group) in enumerate(direct_conflicts.items(), start=1):
    print(f"\n{number}. Claim: {claim}")

    for record in group:
        references = [
            (item[2], item[3])
            for evidence_set in record["evidence"]
            for item in evidence_set
        ]

        print(
            "   ID:", record["id"],
            "| Label:", record["label"],
            "| Evidence references:", references,
        )

Groups containing both SUPPORTS and REFUTES: 3

1. Claim: Janet Leigh was a person.
   ID: 146011 | Label: SUPPORTS | Evidence references: [('Janet_Leigh', 0), ('Janet_Leigh', 2), ('Janet_Leigh', 0), ('Janet_Leigh', 2)]
   ID: 144161 | Label: NOT ENOUGH INFO | Evidence references: [(None, None)]
   ID: 37792 | Label: SUPPORTS | Evidence references: [('Janet_Leigh', 10), ('Janet_Leigh', 12), ('Janet_Leigh', 16), ('Janet_Leigh', 2)]
   ID: 129419 | Label: REFUTES | Evidence references: [('Janet_Leigh', 0)]

2. Claim: The Hundred Years' War includes the Civil War.
   ID: 54077 | Label: REFUTES | Evidence references: [("Hundred_Years'_War", 0), ('American_Civil_War', 0)]
   ID: 519 | Label: SUPPORTS | Evidence references: [("Hundred_Years'_War", 22)]

3. Claim: An island is part of the ABC Islands.
   ID: 139910 | Label: SUPPORTS | Evidence references: [('ABC_islands_-LRB-Lesser_Antilles-RRB-', 0), ('ABC_islands_-LRB-Lesser_Antilles-RRB-', 1), ('ABC_islands_-LRB-Lesser_Antilles-RRB-', 2), 

#### Direct Conflict Result

Three duplicate claim groups contain both `SUPPORTS` and `REFUTES` labels.

The claims appear to include possible annotation inconsistency and ambiguity. However, Wikipedia page names and sentence numbers alone are insufficient to determine the correct labels.

The referenced evidence sentences will therefore be resolved and inspected before reaching a conclusion. The raw records will remain unchanged.

## 5. Evidence Annotation Audit

### 5.1 Evidence Structure

This check validates the nested FEVER evidence format and confirms that verifiable claims contain Wikipedia references while `NOT ENOUGH INFO` claims contain null references.

In [9]:
invalid_groups = []
invalid_items = []
partial_null_references = []

evidence_group_count = 0
evidence_item_count = 0
wiki_reference_count = 0
null_reference_count = 0

verifiable_without_evidence = []
nei_with_wikipedia_evidence = []

for record in records:
    has_wikipedia_reference = False
    evidence_group_count += len(record["evidence"])

    for group_index, group in enumerate(record["evidence"]):
        if not isinstance(group, list) or not group:
            invalid_groups.append((record["id"], group_index))
            continue

        evidence_item_count += len(group)

        for item_index, item in enumerate(group):
            if not isinstance(item, list) or len(item) != 4:
                invalid_items.append((record["id"], group_index, item_index))
                continue

            page, sentence_id = item[2], item[3]

            if page is None and sentence_id is None:
                null_reference_count += 1
            elif isinstance(page, str) and isinstance(sentence_id, int):
                wiki_reference_count += 1
                has_wikipedia_reference = True
            else:
                partial_null_references.append(
                    (record["id"], page, sentence_id)
                )

    if record["label"] == "NOT ENOUGH INFO":
        if has_wikipedia_reference:
            nei_with_wikipedia_evidence.append(record["id"])
    elif not has_wikipedia_reference:
        verifiable_without_evidence.append(record["id"])

print("Evidence groups:", evidence_group_count)
print("Evidence items:", evidence_item_count)
print("Wikipedia references:", wiki_reference_count)
print("Null references:", null_reference_count)
print("Invalid or empty groups:", len(invalid_groups))
print("Invalid evidence items:", len(invalid_items))
print("Partial null references:", len(partial_null_references))
print("Verifiable records without evidence:", len(verifiable_without_evidence))
print("NEI records with Wikipedia evidence:", len(nei_with_wikipedia_evidence))

Evidence groups: 31391
Evidence items: 37566
Wikipedia references: 28625
Null references: 8941
Invalid or empty groups: 0
Invalid evidence items: 0
Partial null references: 0
Verifiable records without evidence: 0
NEI records with Wikipedia evidence: 0


#### Result

**PASS:** All 31,391 evidence groups and 37,566 evidence items follow the expected FEVER structure.

The dataset contains 28,625 Wikipedia evidence references and 8,941 null references. Every verifiable record contains Wikipedia evidence, while no `NOT ENOUGH INFO` record contains a Wikipedia reference.

No invalid groups, malformed items or partially null references were found.

### 5.2 Evidence Reference Inventory

This step collects the unique Wikipedia pages and sentences referenced by the complete development dataset and by the 54 conflicting duplicate groups.

In [10]:
all_references = {
    (item[2], item[3])
    for record in records
    for evidence_set in record["evidence"]
    for item in evidence_set
    if item[2] is not None
}

conflict_references = {
    (item[2], item[3])
    for group in conflicting_claims.values()
    for record in group
    for evidence_set in record["evidence"]
    for item in evidence_set
    if item[2] is not None
}

all_page_titles = {page for page, _ in all_references}
conflict_page_titles = {page for page, _ in conflict_references}

print("Wikipedia reference occurrences:", wiki_reference_count)
print("Unique page-sentence references:", len(all_references))
print(
    "Repeated reference occurrences:",
    wiki_reference_count - len(all_references),
)
print("Unique referenced pages:", len(all_page_titles))
print("Conflict-group references:", len(conflict_references))
print("Conflict-group pages:", len(conflict_page_titles))

Wikipedia reference occurrences: 28625
Unique page-sentence references: 7100
Repeated reference occurrences: 21525
Unique referenced pages: 2892
Conflict-group references: 84
Conflict-group pages: 57


#### Result

The 28,625 Wikipedia reference occurrences reduce to 7,100 unique page-sentence references across 2,892 pages.

The conflicting duplicate groups reference 84 unique sentences from 57 pages. Repeated references are expected because multiple records and annotation sets can use the same evidence.

## 6. Wikipedia Resolution Audit

### 6.1 Referenced Page Resolution

To investigate the conflicting labels, this step retrieves the actual Wikipedia sentences referenced by the records. The same scan also retains the other required FEVER pages so the large Wikipedia collection does not need to be scanned twice.

In [11]:
resolved_pages = {}
duplicate_page_ids = set()

valid_wiki_records = 0
blank_wiki_lines = 0
invalid_wiki_lines = 0
first_invalid_wiki_line = None

for file_number, path in enumerate(wiki_files, start=1):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                blank_wiki_lines += 1
                continue

            try:
                page = json.loads(line)
            except json.JSONDecodeError as error:
                invalid_wiki_lines += 1

                if first_invalid_wiki_line is None:
                    first_invalid_wiki_line = (
                        path.name,
                        line_number,
                        str(error),
                    )
                continue

            valid_wiki_records += 1
            page_id = page.get("id")

            if page_id in all_page_titles:
                if page_id in resolved_pages:
                    duplicate_page_ids.add(page_id)
                else:
                    resolved_pages[page_id] = page

    if file_number % 10 == 0 or file_number == len(wiki_files):
        print(
            f"Scanned {file_number}/{len(wiki_files)} files; "
            f"resolved {len(resolved_pages)}/{len(all_page_titles)} pages"
        )

missing_page_titles = all_page_titles - set(resolved_pages)
missing_conflict_pages = conflict_page_titles - set(resolved_pages)

print("\nValid Wikipedia records:", valid_wiki_records)
print("Blank Wikipedia lines:", blank_wiki_lines)
print("Invalid Wikipedia lines:", invalid_wiki_lines)
print("Requested pages:", len(all_page_titles))
print("Resolved pages:", len(resolved_pages))
print("Missing pages:", len(missing_page_titles))
print("Duplicate requested page IDs:", len(duplicate_page_ids))
print("Missing conflict-group pages:", len(missing_conflict_pages))

if first_invalid_wiki_line:
    print("First invalid line:", first_invalid_wiki_line)

if missing_page_titles:
    print("First 10 missing pages:", sorted(missing_page_titles)[:10])

Scanned 10/109 files; resolved 202/2892 pages
Scanned 20/109 files; resolved 486/2892 pages
Scanned 30/109 files; resolved 762/2892 pages
Scanned 40/109 files; resolved 1000/2892 pages
Scanned 50/109 files; resolved 1290/2892 pages
Scanned 60/109 files; resolved 1484/2892 pages
Scanned 70/109 files; resolved 1737/2892 pages
Scanned 80/109 files; resolved 1995/2892 pages
Scanned 90/109 files; resolved 2298/2892 pages
Scanned 100/109 files; resolved 2666/2892 pages
Scanned 109/109 files; resolved 2860/2892 pages

Valid Wikipedia records: 5416537
Blank Wikipedia lines: 0
Invalid Wikipedia lines: 0
Requested pages: 2892
Resolved pages: 2860
Missing pages: 32
Duplicate requested page IDs: 0
Missing conflict-group pages: 0
First 10 missing pages: ["1974_Cypriot_coup_d'état", 'Bañuela', 'Björk', 'Café_Society_-LRB-film-RRB-', 'Cléopâtre', 'Curaçao', 'Dulé_Hill', 'Eugène_Ionesco', 'Édouard_Séguin', 'Jacques_Clément']


#### Wikipedia Scan Result

**PASS WITH WARNING:** All 5,416,537 Wikipedia records are valid JSON, with no blank or malformed lines.

Of the 2,892 requested pages, 2,860 were resolved and 32 were not matched. The missing titles mainly contain accented characters, suggesting a possible Unicode-normalization mismatch that requires later investigation.

All 57 pages referenced by the conflicting duplicate claims were successfully resolved, so the duplicate investigation can continue.

### 6.2 Evidence Sentence Resolution

This step converts the resolved Wikipedia page and sentence-number references into the evidence text used by the duplicate investigation.

In [12]:
page_sentences = {}

for page_id, page in resolved_pages.items():
    sentences = {}

    for raw_line in page["lines"].splitlines():
        parts = raw_line.split("\t")

        if len(parts) >= 2:
            try:
                sentence_id = int(parts[0])
            except ValueError:
                continue

            sentences[sentence_id] = parts[1]

    page_sentences[page_id] = sentences

resolved_evidence = {}
missing_page_references = set()
missing_sentence_references = set()

for page_id, sentence_id in all_references:
    if page_id not in page_sentences:
        missing_page_references.add((page_id, sentence_id))
    elif sentence_id not in page_sentences[page_id]:
        missing_sentence_references.add((page_id, sentence_id))
    else:
        resolved_evidence[(page_id, sentence_id)] = (
            page_sentences[page_id][sentence_id]
        )

missing_conflict_references = (
    conflict_references - set(resolved_evidence)
)

print("Unique evidence references:", len(all_references))
print("Resolved evidence sentences:", len(resolved_evidence))
print("References with missing pages:", len(missing_page_references))
print("References with missing sentence IDs:", len(missing_sentence_references))
print("Conflict references:", len(conflict_references))
print(
    "Resolved conflict references:",
    len(conflict_references & set(resolved_evidence)),
)
print(
    "Missing conflict references:",
    len(missing_conflict_references),
)

Unique evidence references: 7100
Resolved evidence sentences: 7028
References with missing pages: 72
References with missing sentence IDs: 0
Conflict references: 84
Resolved conflict references: 84
Missing conflict references: 0


#### Evidence Resolution Result

**PASS FOR CONFLICT INVESTIGATION:** All 84 evidence references used by the conflicting duplicate claims were successfully resolved into Wikipedia sentences.

No conflicting claim is missing its referenced evidence. The 72 unresolved references belong to other records and will be examined later with the 32 unmatched page titles.

## 7. Content-Quality Audit

### 7.1 Direct Label Conflicts

This review examines the actual Wikipedia evidence for the three duplicate claim groups containing both `SUPPORTS` and `REFUTES` labels.

Evidence sets are kept separate because every sentence within one set may be required to justify a label.

In [13]:
for number, (claim, group) in enumerate(
    direct_conflicts.items(),
    start=1,
):
    print(f"\n{'=' * 80}")
    print(f"{number}. CLAIM: {claim}")

    for record in group:
        print(f"\nID {record['id']} — {record['label']}")

        shown_groups = set()

        for evidence_set in record["evidence"]:
            references = tuple(
                (item[2], item[3])
                for item in evidence_set
                if item[2] is not None
            )

            if not references or references in shown_groups:
                continue

            shown_groups.add(references)
            print("  Evidence set:")

            for reference in references:
                print(
                    f"    {reference}: "
                    f"{resolved_evidence[reference]}"
                )

        if not shown_groups:
            print("  No Wikipedia evidence")


1. CLAIM: Janet Leigh was a person.

ID 146011 — SUPPORTS
  Evidence set:
    ('Janet_Leigh', 0): Janet Leigh -LRB- born Jeanette Helen Morrison ; July 6 , 1927 -- October 3 , 2004 -RRB- was an American actress , singer , dancer and author .
  Evidence set:
    ('Janet_Leigh', 2): She was the first wife of actor Tony Curtis and the mother of Kelly Curtis and Jamie Lee Curtis .

ID 144161 — NOT ENOUGH INFO
  No Wikipedia evidence

ID 37792 — SUPPORTS
  Evidence set:
    ('Janet_Leigh', 10): Her highly publicized marriage to Curtis ended in divorce in 1962 , and after starring in The Manchurian Candidate that same year , Leigh scaled back her career .
  Evidence set:
    ('Janet_Leigh', 12): She also wrote four books between 1984 and 2002 , including two novels .
  Evidence set:
    ('Janet_Leigh', 16): Among her survivors was her husband of 42 years , Robert Brandt .
  Evidence set:
    ('Janet_Leigh', 2): She was the first wife of actor Tony Curtis and the mother of Kelly Curtis and J

#### Direct Conflict Assessment

| Claim | Assessment | Reason |
|---|---|---|
| Janet Leigh was a person. | Likely annotation error | The evidence identifies Janet Leigh as an actress, singer, dancer, author, wife and mother. This supports the claim, while the `REFUTES` label contradicts its own evidence. |
| The Hundred Years' War includes the Civil War. | Ambiguous claim | “The Civil War” could refer to the American Civil War or the Castilian Civil War. The records use different interpretations. |
| An island is part of the ABC Islands. | Likely annotation error | The evidence explicitly identifies Aruba, Curaçao and Bonaire as the ABC islands. The evidence attached to `REFUTES` does not refute the claim. |

These records will remain unchanged in the raw dataset. They should be flagged when deciding which records are eligible for experiments.

### 7.2 Conflict Handling Decision

The audit identified 54 exact duplicate claim groups with conflicting labels. Exhaustively relabelling these records is outside the scope of a structural data audit and would introduce subjective decisions.

Therefore:

- The raw FEVER records and labels remain unchanged.
- All records belonging to the 54 conflicting groups are flagged.
- These records will be excluded from the proposed clean evaluation subset.
- Label distribution will be recalculated after exclusion.
- The three direct conflicts examined above are retained as documented examples.

In [14]:
conflicting_record_ids = {
    record["id"]
    for group in conflicting_claims.values()
    for record in group
}

audit_flags = {
    record["id"]: (
        ["conflicting_duplicate_label"]
        if record["id"] in conflicting_record_ids
        else []
    )
    for record in records
}

conflict_filtered_records = [
    record
    for record in records
    if record["id"] not in conflicting_record_ids
]

remaining_label_counts = Counter(
    record["label"]
    for record in conflict_filtered_records
)

print("Conflicting claim groups flagged:", len(conflicting_claims))
print("Records flagged:", len(conflicting_record_ids))
print("Original records:", len(records))
print("Records remaining:", len(conflict_filtered_records))
print(
    "Percentage excluded:",
    f"{len(conflicting_record_ids) / len(records):.2%}",
)

print("\nRemaining label distribution:")
for label, count in sorted(remaining_label_counts.items()):
    print(
        f"{label}: {count} "
        f"({count / len(conflict_filtered_records):.2%})"
    )

Conflicting claim groups flagged: 54
Records flagged: 124
Original records: 19998
Records remaining: 19874
Percentage excluded: 0.62%

Remaining label distribution:
NOT ENOUGH INFO: 6609 (33.25%)
REFUTES: 6619 (33.30%)
SUPPORTS: 6646 (33.44%)


#### Conflict-Flagging Result

The 54 conflicting duplicate groups contain 124 records, representing 0.62% of the development dataset.

After flagging these records, 19,874 records remain. The three labels remain approximately balanced, so excluding the flagged conflicts would not materially distort the class distribution.

The raw dataset remains unchanged.

### 7.3 Unicode Resolution of Unmatched Wikipedia Pages

The 32 unmatched Wikipedia titles mainly contain accented characters. This check compares Unicode-normalized page titles to determine whether the pages are present under an equivalent character representation.

In [15]:
import unicodedata

normalized_targets = defaultdict(list)

for title in missing_page_titles:
    normalized_title = unicodedata.normalize("NFC", title)
    normalized_targets[normalized_title].append(title)

normalization_matches = {}
normalization_collisions = set()

for file_number, path in enumerate(wiki_files, start=1):
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            page = json.loads(line)
            page_id = page.get("id")

            if not isinstance(page_id, str):
                continue

            normalized_page_id = unicodedata.normalize("NFC", page_id)

            for requested_title in normalized_targets.get(
                normalized_page_id,
                [],
            ):
                if requested_title in normalization_matches:
                    normalization_collisions.add(requested_title)
                else:
                    normalization_matches[requested_title] = page

    if file_number % 20 == 0 or file_number == len(wiki_files):
        print(
            f"Scanned {file_number}/{len(wiki_files)} files; "
            f"matched {len(normalization_matches)}/{len(missing_page_titles)}"
        )

still_missing_titles = (
    missing_page_titles - set(normalization_matches)
)

print("\nOriginally missing pages:", len(missing_page_titles))
print("Matched after Unicode normalization:", len(normalization_matches))
print("Still missing:", len(still_missing_titles))
print("Normalization collisions:", len(normalization_collisions))

if still_missing_titles:
    print("Still missing titles:", sorted(still_missing_titles))

Scanned 20/109 files; matched 4/32
Scanned 40/109 files; matched 8/32
Scanned 60/109 files; matched 12/32
Scanned 80/109 files; matched 18/32
Scanned 100/109 files; matched 28/32
Scanned 109/109 files; matched 32/32

Originally missing pages: 32
Matched after Unicode normalization: 32
Still missing: 0
Normalization collisions: 0


#### Unicode Resolution Result

**PASS:** All 32 previously unmatched Wikipedia pages were found after Unicode NFC normalization.

No pages remain missing, and no normalization collisions were detected. Therefore, the issue was caused by equivalent accented characters being encoded differently rather than missing Wikipedia data.

The raw files remain unchanged. Future processing should normalize page identifiers before matching them.

In [16]:
resolved_pages.update(normalization_matches)

for requested_title, page in normalization_matches.items():
    sentences = {}

    for raw_line in page["lines"].splitlines():
        parts = raw_line.split("\t")

        if len(parts) >= 2:
            try:
                sentence_id = int(parts[0])
            except ValueError:
                continue

            sentences[sentence_id] = parts[1]

    page_sentences[requested_title] = sentences

resolved_evidence = {}
missing_page_references = set()
missing_sentence_references = set()

for reference in all_references:
    page_id, sentence_id = reference

    if page_id not in page_sentences:
        missing_page_references.add(reference)
    elif sentence_id not in page_sentences[page_id]:
        missing_sentence_references.add(reference)
    else:
        resolved_evidence[reference] = (
            page_sentences[page_id][sentence_id]
        )

print("Unique evidence references:", len(all_references))
print("Resolved evidence sentences:", len(resolved_evidence))
print("Missing page references:", len(missing_page_references))
print("Missing sentence references:", len(missing_sentence_references))

Unique evidence references: 7100
Resolved evidence sentences: 7100
Missing page references: 0
Missing sentence references: 0


#### Final Evidence Resolution Result

**PASS:** All 7,100 unique Wikipedia evidence references were successfully resolved.

No referenced pages or sentence IDs remain missing. Unicode normalization resolved the 32 previously unmatched page titles without changing the raw data.

## 8. Experiment-Focused Data Analysis

### 8.1 Claim Length

This check measures claim length overall and by label. Word counts use whitespace splitting, so they are approximate lengths, not model token counts.

In [17]:
from statistics import median

def length_summary(values):
    ordered = sorted(values)
    return {
        "min": ordered[0],
        "median": median(ordered),
        "p95": ordered[int(0.95 * (len(ordered) - 1))],
        "max": ordered[-1],
    }

for label in ["ALL", "SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
    subset = records if label == "ALL" else [
        record for record in records if record["label"] == label
    ]
    word_lengths = [len(record["claim"].split()) for record in subset]
    character_lengths = [len(record["claim"]) for record in subset]

    print(
        label,
        "| records:", len(subset),
        "| words:", length_summary(word_lengths),
        "| characters:", length_summary(character_lengths),
    )

longest = max(records, key=lambda record: len(record["claim"].split()))
print("\nLongest claim by word count:")
print("ID:", longest["id"], "| Label:", longest["label"])
print(longest["claim"])

ALL | records: 19998 | words: {'min': 2, 'median': 8.0, 'p95': 14, 'max': 36} | characters: {'min': 13, 'median': 46.0, 'p95': 84, 'max': 219}
SUPPORTS | records: 6666 | words: {'min': 2, 'median': 8.0, 'p95': 14, 'max': 31} | characters: {'min': 13, 'median': 45.0, 'p95': 85, 'max': 200}
REFUTES | records: 6666 | words: {'min': 3, 'median': 8.0, 'p95': 14, 'max': 31} | characters: {'min': 14, 'median': 46.0, 'p95': 81, 'max': 176}
NOT ENOUGH INFO | records: 6666 | words: {'min': 2, 'median': 8.0, 'p95': 14, 'max': 36} | characters: {'min': 15, 'median': 47.0, 'p95': 86, 'max': 219}

Longest claim by word count:
ID: 111251 | Label: NOT ENOUGH INFO
Tremont Street Subway served a light rail station on the MBTA Green Line system, and is located on the southeast corner of Boston Common at the intersection of Boylston Street and Tremont Street called Boylston station.


#### Claim-Length Result

FEVER claims are generally short. The median claim length is 8 words, and 95% of claims are approximately 14 words or shorter. Claim lengths are similar across the three labels.

These are whitespace-based word counts, not model token counts. Evidence length must also be checked before assessing prompt size.

### 8.2 Evidence Complexity

This check measures how many sentences and Wikipedia pages are needed in the smallest complete evidence set for each verifiable claim. It follows the deterministic selection rule used by the pilot baseline.

In [18]:
evidence_stats = []

for record in records:
    if record["label"] == "NOT ENOUGH INFO":
        continue

    sets = [
        tuple(sorted({(item[2], item[3]) for item in group}))
        for group in record["evidence"]
    ]
    selected = min(sets, key=lambda refs: (len(refs), refs))

    evidence_stats.append({
        "label": record["label"],
        "sentence_count": len(selected),
        "page_count": len({page for page, _ in selected}),
        "evidence_words": sum(
            len(resolved_evidence[reference].split())
            for reference in selected
        ),
    })

for label in ["ALL", "SUPPORTS", "REFUTES"]:
    subset = evidence_stats if label == "ALL" else [
        row for row in evidence_stats if row["label"] == label
    ]

    print(
        label,
        "| records:", len(subset),
        "| one sentence:", sum(
            row["sentence_count"] == 1 for row in subset
        ),
        "| multiple sentences:", sum(
            row["sentence_count"] > 1 for row in subset
        ),
        "| multiple pages:", sum(
            row["page_count"] > 1 for row in subset
        ),
        "| evidence words:", length_summary(
            [row["evidence_words"] for row in subset]
        ),
    )

ALL | records: 13332 | one sentence: 12128 | multiple sentences: 1204 | multiple pages: 1060 | evidence words: {'min': 4, 'median': 28.0, 'p95': 65, 'max': 525}
SUPPORTS | records: 6666 | one sentence: 5991 | multiple sentences: 675 | multiple pages: 597 | evidence words: {'min': 4, 'median': 28.0, 'p95': 67, 'max': 342}
REFUTES | records: 6666 | one sentence: 6137 | multiple sentences: 529 | multiple pages: 463 | evidence words: {'min': 4, 'median': 28.0, 'p95': 64, 'max': 525}


#### Evidence-Complexity Result

Most verifiable claims use one evidence sentence. However, 1,204 claims require multiple sentences, and 1,060 use evidence from multiple Wikipedia pages.

Selected evidence has a median length of 28 words and a 95th-percentile length of 65 words. The longest selected evidence is 525 words. These are word counts, not model token counts.

### 8.3 Full Prompt Length

This check measures the text sent to the model using the project's existing evidence-selection rule and prompt format. Word counts are approximate and do not measure model tokens.

In [19]:
from llm_fact_checker.data_prep.fever import select_evidence_set
from llm_fact_checker.prompts import SYSTEM_PROMPT, build_user_prompt

prompt_lengths = []

for record in records:
    if record["label"] == "NOT ENOUGH INFO":
        continue

    selected = select_evidence_set(record["evidence"])
    assert selected is not None

    evidence = [
        {
            **reference,
            "text": resolved_evidence[
                (reference["page"], reference["sentence_id"])
            ],
        }
        for reference in selected
    ]

    prompt = SYSTEM_PROMPT + "\n" + build_user_prompt(
        record["claim"], evidence
    )
    prompt_lengths.append(
        (record["id"], record["label"], len(prompt.split()), len(prompt))
    )

word_counts = [row[2] for row in prompt_lengths]
character_counts = [row[3] for row in prompt_lengths]
longest = max(prompt_lengths, key=lambda row: row[2])

print("Prompts measured:", len(prompt_lengths))
print("Prompt words:", length_summary(word_counts))
print("Prompt characters:", length_summary(character_counts))
print("Prompts over 256 words:", sum(n > 256 for n in word_counts))
print("Prompts over 512 words:", sum(n > 512 for n in word_counts))
print("Longest prompt ID and label:", longest[:2])

2026-10-11 23:54:45.573 | INFO     | llm_fact_checker.config:<module>:11 - PROJ_ROOT path is: /Users/shanusharma/Desktop/UTS/NLP/Project/llm-fact-checker


Prompts measured: 13332
Prompt words: {'min': 70, 'median': 96.0, 'p95': 138, 'max': 706}
Prompt characters: {'min': 471, 'median': 617.0, 'p95': 857, 'max': 4099}
Prompts over 256 words: 19
Prompts over 512 words: 1
Longest prompt ID and label: (93952, 'REFUTES')


#### Full-Prompt Length Result

The complete system and user prompts have a median length of 96 words and a 95th-percentile length of 138 words.

Only 19 of 13,332 prompts exceed 256 words, and one exceeds 512 words. The longest prompt is 706 words. Word counts are approximate and do not establish model token usage or truncation behaviour.

### 8.4 Normalized Claim Duplicates

This check finds claim texts that become identical after Unicode, capitalization and whitespace normalization. It flags possible repetition; it does not change claims or labels.

In [20]:
normalized_claims = defaultdict(list)

for record in records:
    key = " ".join(
        unicodedata.normalize("NFC", record["claim"])
        .casefold()
        .split()
    )
    normalized_claims[key].append(record)

variant_groups = {
    key: group
    for key, group in normalized_claims.items()
    if len({record["claim"] for record in group}) > 1
}

variant_conflicts = {
    key: group
    for key, group in variant_groups.items()
    if len({record["label"] for record in group}) > 1
}

print("Groups with differently written claim text:", len(variant_groups))
print(
    "Records in those groups:",
    sum(len(group) for group in variant_groups.values()),
)
print(
    "Groups also containing conflicting labels:",
    len(variant_conflicts),
)

Groups with differently written claim text: 2
Records in those groups: 4
Groups also containing conflicting labels: 1


#### Normalized-Duplicate Finding

Two groups (four records) contain differently written claim text that becomes identical after normalization. One of these groups has conflicting labels. Their exact wording and evidence references will be inspected before assigning any additional flags.

In [21]:
for number, group in enumerate(variant_groups.values(), start=1):
    print(f"\nGroup {number}")

    for record in group:
        references = [
            (item[2], item[3])
            for evidence_set in record["evidence"]
            for item in evidence_set
            if item[2] is not None
        ]

        print("ID:", record["id"])
        print("Claim:", repr(record["claim"]))
        print("Label:", record["label"])
        print("Evidence references:", references)


Group 1
ID: 161846
Claim: 'Robert Lopez has won an award.'
Label: SUPPORTS
Evidence references: [('Robert_Lopez', 1)]
ID: 161865
Claim: 'Robert Lopez has won an Award.'
Label: SUPPORTS
Evidence references: [('Robert_Lopez', 1)]

Group 2
ID: 94019
Claim: 'Atlanta metropolitan area is located in north Georgia.'
Label: SUPPORTS
Evidence references: [('Atlanta_metropolitan_area', 3)]
ID: 5544
Claim: 'Atlanta metropolitan area is located in North Georgia.'
Label: NOT ENOUGH INFO
Evidence references: []


#### Normalized-Duplicate Review

The Robert Lopez pair differs only in capitalization and has matching labels and evidence.

The Atlanta pair has different labels. Whether “north Georgia” and “North Georgia” mean the same thing requires checking the cited Wikipedia sentence. No additional record is flagged yet.

In [22]:
for group in variant_conflicts.values():
    print("Claim variants:")
    for record in group:
        print(record["id"], repr(record["claim"]), record["label"])

        for evidence_set in record["evidence"]:
            for item in evidence_set:
                if item[2] is not None:
                    reference = (item[2], item[3])
                    print("Evidence:", resolved_evidence[reference])

Claim variants:
94019 'Atlanta metropolitan area is located in north Georgia.' SUPPORTS
Evidence: The Combined Statistical Area spans up to 39 counties in north Georgia and had an estimated 2015 population of 6.3 million people .
5544 'Atlanta metropolitan area is located in North Georgia.' NOT ENOUGH INFO


#### Normalized-Duplicate Result

Two capitalization variants were found. The Robert Lopez pair has matching `SUPPORTS` labels and cites the same evidence.

The Atlanta pair has `SUPPORTS` and `NOT ENOUGH INFO` labels. Its cited sentence refers to a Combined Statistical Area in “north Georgia”; this does not conclusively settle the differently capitalized claim about the Atlanta metropolitan area. IDs `94019` and `5544` are flagged for possible label inconsistency, separately from the 54 exact-text conflict groups.

No raw labels are changed.

## 9. Experiment eligibility

First, count which records have FEVER-supplied evidence and which records need a separate evidence setup. Flags are counted, not removed.

In [23]:
normalized_conflict_ids = {
    record["id"]
    for group in variant_conflicts.values()
    for record in group
}

for label in ("SUPPORTS", "REFUTES", "NOT ENOUGH INFO"):
    group = [record for record in records if record["label"] == label]
    print(
        f"{label}: {len(group)} records | "
        f"exact-text conflict flag: {sum(r['id'] in conflicting_record_ids for r in group)} | "
        f"normalized-text conflict flag: {sum(r['id'] in normalized_conflict_ids for r in group)}"
    )

SUPPORTS: 6666 records | exact-text conflict flag: 20 | normalized-text conflict flag: 1
REFUTES: 6666 records | exact-text conflict flag: 47 | normalized-text conflict flag: 0
NOT ENOUGH INFO: 6666 records | exact-text conflict flag: 57 | normalized-text conflict flag: 1


### Experiment eligibility decision

The main gold-evidence experiment will use the 13,332 SUPPORTS and REFUTES records. FEVER supplies evidence for these records. This tests how the model judges a claim **when given the dataset’s evidence**; it does not test whether the model can find evidence itself.

We will keep flagged records in the main result and also report a second result without them. Flags indicate possible label inconsistencies, not proven errors. Raw data and labels remain unchanged.

The 6,666 NOT ENOUGH INFO records will not be mixed into this gold-evidence experiment because FEVER supplies no gold evidence sentences for them. A separate experiment must define how evidence is retrieved and apply that method consistently to all three labels.

## 10. FEVER data audit summary

The development file contains 19,998 valid claims: 6,666 each labelled SUPPORTS, REFUTES, and NOT ENOUGH INFO. Its records have a consistent structure. All 7,100 unique Wikipedia evidence references were resolved after Unicode normalization.

We flagged 54 groups where identical claim text has conflicting labels (124 records). We also found one conflicting pair that differs only in capitalization. These are **possible inconsistencies**, not proof that every flagged label is wrong. The raw data remains unchanged.

For the gold-evidence experiment, we will use the 13,332 SUPPORTS and REFUTES records. We will report the main result with flagged records included and a second result without them. NOT ENOUGH INFO requires a separate, consistently applied evidence-retrieval setup because FEVER provides it with no gold evidence sentences.

The audit is complete. These findings define how we prepare and interpret the experiments; they do not measure model accuracy.